In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
data_dir = Path("../data")

In [3]:
# md5sum-lite 465e83dbb75c6b6d4b1332d655474a95
# sc2ts_v2_2024-06-06_pr_pp_mp_aph_bps_pango_dated_recombinants_rebar_pangonet_nsl_akm_mns.csv
recomb_file = data_dir / "recombinants.csv"
recomb_df = pd.read_csv(recomb_file)

In [4]:
# Good recombinants
pd.set_option("display.max_rows", 1000)
recomb_df[recomb_df['net_min_supporting_loci_lft_rgt_ge_4']][
    [
        'recombinant',
        'net_min_supporting_loci_lft_rgt_ge_4',
        'net_min_supporting_loci_lft',
        'net_min_supporting_loci_rgt',
        'num_mns_left',
        'mns_left',
        'num_mns_right',
        'mns_right',
    ]
].sort_values(
    by=[
        'num_mns_right',
        'num_mns_left',
    ],
    ascending=False
).head(116)

,recombinant,net_min_supporting_loci_lft_rgt_ge_4,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,num_mns_left,mns_left,num_mns_right,mns_right
53,425807,True,8,61,0,NaN,2,"MNM1,MNM1a"
133,1446986,True,15,34,0,NaN,2,"MNM2,MNM3"
23,1352592,True,5,47,0,NaN,1,MNM3
24,1169218,True,5,44,0,NaN,1,MNM3
25,1375045,True,4,44,0,NaN,1,MNM3
26,1578074,True,5,43,0,NaN,1,MNM3
27,1818909,True,4,43,0,NaN,1,MNM3
29,1702308,True,5,39,0,NaN,1,MNM3
34,1358917,True,8,46,0,NaN,1,MNM3
37,1412252,True,6,46,0,NaN,1,MNM3


In [5]:
num_good_recomb_total = np.sum(recomb_df.net_min_supporting_loci_lft_rgt_ge_4)
num_good_recomb_with_mns = np.sum(
    np.logical_and(
        recomb_df.net_min_supporting_loci_lft_rgt_ge_4,
        np.logical_or(recomb_df.num_mns_left > 0, recomb_df.num_mns_right > 0)
    )
)
perc_good_recomb_with_mns = num_good_recomb_with_mns / num_good_recomb_total

num_bad_recomb_total = np.sum(~recomb_df.net_min_supporting_loci_lft_rgt_ge_4)
num_bad_recomb_with_mns = np.sum(
    np.logical_and(
        ~recomb_df.net_min_supporting_loci_lft_rgt_ge_4,
        np.logical_or(recomb_df.num_mns_left > 0, recomb_df.num_mns_right > 0)
    )
)
perc_bad_recomb_with_mns = num_bad_recomb_with_mns / num_bad_recomb_total

print(f"Good recomb. with MNS: {num_good_recomb_with_mns} / {num_good_recomb_total} ({100 * perc_good_recomb_with_mns:.0f}%)")
print(f" Bad recomb. with MNS: {num_bad_recomb_with_mns} / {num_bad_recomb_total} ({100 * perc_bad_recomb_with_mns:.0f}%)")

Good recomb. with MNS: 115 / 647 (18%)
 Bad recomb. with MNS: 87 / 673 (13%)


In [6]:
# Bad recombinants
pd.set_option("display.max_rows", 1000)
recomb_df[~recomb_df['net_min_supporting_loci_lft_rgt_ge_4']][
    [
        'recombinant',
        'net_min_supporting_loci_lft_rgt_ge_4',
        'net_min_supporting_loci_lft',
        'net_min_supporting_loci_rgt',
        'num_mns_left',
        'mns_left',
        'num_mns_right',
        'mns_right',
    ]
].sort_values(
    by=[
        'num_mns_left',
        'num_mns_right',
    ],
    ascending=False
).head(88)

,recombinant,net_min_supporting_loci_lft_rgt_ge_4,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,num_mns_left,mns_left,num_mns_right,mns_right
537,603979,False,5,1,2,"MNM1,MNM1a",0,NaN
762,708357,False,5,0,2,"MNM1,MNM1a",0,NaN
1290,1790988,False,103,2,2,"MNM3,mnS3",0,NaN
1207,1257878,False,51,1,1,MNM3,2,"MNM7,MNM7a"
1209,1190264,False,7,1,1,MNM12,2,"MNM7,MNM7a"
1220,979469,False,9,2,1,MNM12,2,"MNM7,MNM7a"
0,1438485,False,2,55,1,mnS1,1,MNM3
1234,259323,False,13,2,1,mnS2,1,MNM7a
2,752970,False,2,19,1,mnS1,0,NaN
7,1002427,False,2,38,1,mnS1,0,NaN


### Counts of each MNS by QC status

In [7]:
qc_pass = recomb_df.net_min_supporting_loci_lft_rgt_ge_4
mns = pd.concat([
    recomb_df[["recombinant", "mns_left"]].rename(columns={"mns_left": "mns"}),
    recomb_df[["recombinant", "mns_right"]].rename(columns={"mns_right": "mns"}),
]).dropna()
mns["mns"] = mns["mns"].str.split(",")
mns = mns.explode("mns")
mns["qc_pass"] = qc_pass.to_numpy()[recomb_df.index.get_indexer(mns.index)]
mns.groupby(["mns", "qc_pass"]).size().unstack(fill_value=0).sort_values(True, ascending=False)

qc_pass,False,True
mns,,
MNM3,16,86
mnS3,4,18
MNM2,4,6
MNM1a,5,3
MNM1,2,1
MNM4,6,1
MNM6,0,1
mnS1,9,1
MNM11,1,0


### Sensitivity: discount one locus per supporting MNS

How many QC-passing recombinants would fall below the cutoff of 4 net supporting loci?

In [8]:
net_lft_no_mns = recomb_df.net_min_supporting_loci_lft - recomb_df.num_mns_left
net_rgt_no_mns = recomb_df.net_min_supporting_loci_rgt - recomb_df.num_mns_right
drops_below = qc_pass & ((net_lft_no_mns < 4) | (net_rgt_no_mns < 4))
print(f"QC-passing recombinants failing QC if MNS are discounted: {np.sum(drops_below)} / {np.sum(qc_pass)}")
recomb_df[drops_below][
    [
        'recombinant',
        'net_min_supporting_loci_lft',
        'net_min_supporting_loci_rgt',
        'mns_left',
        'mns_right',
        'parent_left_pango',
        'parent_right_pango',
    ]
]

QC-passing recombinants failing QC if MNS are discounted: 7 / 647


,recombinant,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,mns_left,mns_right,parent_left_pango,parent_right_pango
253,25956,7,4,NaN,mnS3,B.39,B.1.1
814,1344320,5,4,NaN,MNM2,BA.1.1,BA.1.1
1154,1532072,17,4,NaN,MNM3,BF.4,BA.2.3
1157,1762541,21,4,NaN,MNM3,BA.5.3.5,BA.2
1164,1536421,11,4,NaN,MNM3,BA.5.2.1,BA.2
1168,1670717,14,4,NaN,MNM3,BA.5.2.1,BA.2
1285,112685,40,4,NaN,mnS3,B.1.177,B.1.1.7


### mnS1 (A507T-T508C-G509A)

In [9]:
has_mns1 = (
    recomb_df.mns_left.str.split(",").apply(lambda x: isinstance(x, list) and "mnS1" in x)
    | recomb_df.mns_right.str.split(",").apply(lambda x: isinstance(x, list) and "mnS1" in x)
)
print(f"Recombinants with mnS1: {np.sum(has_mns1)}")
print(f"Of which failed QC: {np.sum(has_mns1 & ~qc_pass)}")

Recombinants with mnS1: 10
Of which failed QC: 9


### MNM7 and MNM7a (27875-27883)

In [10]:
has_mnm7 = (
    recomb_df.mns_left.str.split(",").apply(lambda x: isinstance(x, list) and bool({"MNM7", "MNM7a"} & set(x)))
    | recomb_df.mns_right.str.split(",").apply(lambda x: isinstance(x, list) and bool({"MNM7", "MNM7a"} & set(x)))
)
print(f"Recombinants with MNM7 or MNM7a: {np.sum(has_mnm7)}")
print(f"Of which failed QC: {np.sum(has_mnm7 & ~qc_pass)}")

Recombinants with MNM7 or MNM7a: 42
Of which failed QC: 42


### MNM3 (27382-27384) supporting the right parent in QC-passing recombinants

In [11]:
has_mnm3_right = qc_pass & recomb_df.mns_right.str.split(",").apply(lambda x: isinstance(x, list) and "MNM3" in x)
print(f"QC-passing recombinants with MNM3 supporting the right parent: {np.sum(has_mnm3_right)}")
recomb_df[has_mnm3_right].parent_right_pango.value_counts()

QC-passing recombinants with MNM3 supporting the right parent: 84


parent_right_pango
BA.2          31
BA.2.9         8
BA.4.6         5
BA.2.12.1      3
BA.4           3
XBB.1.5        3
BA.2.3         2
XBB.1          2
BA.2.5         2
BA.2.10        2
BA.4.1         2
BA.2.48        1
BN.1.3.8       1
BA.2.10.1      1
CV.1           1
BN.1.1         1
CH.1.1.1       1
BA.2.37        1
BA.2.13        1
XBB.1.5.18     1
XBB            1
XBB.2.3        1
BA.4.6.5       1
BA.2.22        1
CH.1.1.2       1
XBB.1.5.16     1
HT.1           1
CJ.1           1
CJ.1.2         1
BN.1.3         1
FY.1           1
CH.1.1.24      1
Name: count, dtype: int64